In [ ]:
!pip install git+https://github.com/AI4Finance-Foundation/FinRL.git


  Cloning https://github.com/AI4Finance-Foundation/FinRL.git to c:\users\yep!\appdata\local\temp\pip-req-build-6moq2mam
  Resolved https://github.com/AI4Finance-Foundation/FinRL.git to commit 8cf3cacc6f570d26b430e403ea522c8fe9e6876a
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Cloning https://github.com/AI4Finance-Foundation/ElegantRL.git to c:\users\yep!\appdata\local\temp\pip-install-4dwgkfb1\elegantrl_52fa41ca52df47a788c2727f94aa422a
  Resolved https://github.com/AI4Finance-Foundation/ElegantRL.git to commit 3bdc958c8e624b61a9e661832b01fef816924f61
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Gett

  Running command git clone --filter=blob:none --quiet https://github.com/AI4Finance-Foundation/FinRL.git 'C:\Users\YEP!\AppData\Local\Temp\pip-req-build-6moq2mam'
  Running command git clone --filter=blob:none --quiet https://github.com/AI4Finance-Foundation/ElegantRL.git 'C:\Users\YEP!\AppData\Local\Temp\pip-install-4dwgkfb1\elegantrl_52fa41ca52df47a788c2727f94aa422a'

[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: C:\Users\YEP!\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip
ERROR: Package 'finrl' requires a different Python: 3.10.0 not in '<3.12,>=3.11'


In [ ]:
    # ## install finrl library
    !pip install wrds
    !pip install swig
    !apt-get update -y -qq && apt-get install -y -qq cmake libopenmpi-dev python3-dev zlib1g-dev libgl1-mesa-glx swig
    !pip install finrl


[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: C:\Users\YEP!\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: C:\Users\YEP!\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip
'apt-get' is not recognized as an internal or external command,
operable program or batch file.



[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: C:\Users\YEP!\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip


In [4]:
import warnings
warnings.filterwarnings("ignore")

In [10]:
pip show finrl


Name: finrl
Version: 0.3.6
Summary: FinRL: Financial Reinforcement Learning Framework. Version 0.3.5 notes: stable version, code refactoring, more tutorials, clear documentation
Home-page: 
Author: Hongyang Yang, Xiaoyang Liu
Author-email: 
License: MIT
Location: c:\Users\YEP!\AppData\Local\Programs\Python\Python311\Lib\site-packages
Requires: alpaca-py, alpaca-trade-api, ccxt, elegantrl, exchange-calendars, jqdatasdk, pyfolio, pyportfolioopt, ray, scikit-learn, selenium, stable-baselines3, stockstats, webdriver-manager, wrds, yfinance
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [11]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
# matplotlib.use('Agg')
import datetime

%matplotlib inline
from finrl.config_tickers import DOW_30_TICKER
from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.preprocessor.preprocessors import FeatureEngineer, data_split
from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv
from finrl.agents.stablebaselines3.models import DRLAgent,DRLEnsembleAgent
from finrl.plot import backtest_stats, backtest_plot, get_daily_return, get_baseline

from pprint import pprint

import sys
sys.path.append("../FinRL-Library")

import itertools

In [12]:
import os
from finrl.main import check_and_make_directories
from finrl.config import (
    DATA_SAVE_DIR,
    TRAINED_MODEL_DIR,
    TENSORBOARD_LOG_DIR,
    RESULTS_DIR,
    INDICATORS,
    TRAIN_START_DATE,
    TRAIN_END_DATE,
    TEST_START_DATE,
    TEST_END_DATE,
    TRADE_START_DATE,
    TRADE_END_DATE,
)

check_and_make_directories([DATA_SAVE_DIR, TRAINED_MODEL_DIR, TENSORBOARD_LOG_DIR, RESULTS_DIR])

In [45]:

#Saved the preprocessed historical_data.csv as data.csv

df = pd.read_csv('C:/Users/YEP!/Desktop/FinRL_Project/data.csv')

In [47]:
#Data Exploration
print("First 5 rows of the dataset:")
print(df.head())

First 5 rows of the dataset:
         date      close       high        low       open  volume     tic  \
0  2004-01-01   0.105000   0.105000   0.105000   0.105000       0  DYA.TO   
1  2004-01-01   1.378000   1.378000   1.378000   1.378000       0  EDT.TO   
2  2004-01-01   0.300000   0.300000   0.300000   0.300000       0  TSK.TO   
3  2004-01-02   0.489625   0.489625   0.489625   0.489625       0  AAB.TO   
4  2004-01-02  22.070263  22.211644  21.817266  21.899118  413000  ABX.TO   

   day  daily_return  
0    3           NaN  
1    3     12.123810  
2    3     -0.782293  
3    4      0.632084  
4    4     44.075824  


In [48]:
print(df.isnull().sum()) 

date            0
close           0
high            0
low             0
open            0
volume          0
tic             0
day             0
daily_return    1
dtype: int64


In [49]:
# Convert 'date' column to datetime if it is not already
df['date'] = pd.to_datetime(df['date']) 

# *Step 1: Data Cleaning*
df = df.sort_values(["date", "tic"], ignore_index=True)

# Factorize the date index
df.index = df.date.factorize()[0]
df

,date,close,high,low,open,volume,tic,day,daily_return
0,2004-01-01,0.105000,0.105000,0.105000,0.105000,0,DYA.TO,3,NaN
0,2004-01-01,1.378000,1.378000,1.378000,1.378000,0,EDT.TO,3,12.123810
0,2004-01-01,0.300000,0.300000,0.300000,0.300000,0,TSK.TO,3,-0.782293
1,2004-01-02,0.489625,0.489625,0.489625,0.489625,0,AAB.TO,4,0.632084
1,2004-01-02,22.070263,22.211644,21.817266,21.899118,413000,ABX.TO,4,44.075824
...,...,...,...,...,...,...,...,...,...
5280,2024-12-30,20.230000,20.309999,20.150000,20.309999,700,ZWS.TO,0,0.182349
5280,2024-12-30,53.099998,53.360001,52.830002,53.299999,11600,ZWT.TO,0,1.624815
5280,2024-12-30,10.510000,10.520000,10.430000,10.500000,168700,ZWU.TO,0,-0.802072
5280,2024-12-30,42.599998,42.610001,42.599998,42.610001,300,ZXM.TO,0,3.053282


In [50]:
# Drop rows where any of the required columns have NaN values
df = df.dropna(subset=["close", "high", "low", "open", "tic", "day"])

# *Step 3: Add User Defined Features*
df["daily_return"] = df.close.pct_change(1)

In [51]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4066965 entries, 0 to 5280
Data columns (total 9 columns):
 #   Column        Dtype         
---  ------        -----         
 0   date          datetime64[ns]
 1   close         float64       
 2   high          float64       
 3   low           float64       
 4   open          float64       
 5   volume        int64         
 6   tic           object        
 7   day           int64         
 8   daily_return  float64       
dtypes: datetime64[ns](1), float64(5), int64(2), object(1)
memory usage: 310.3+ MB


In [ ]:
#Selecting the indicators for the risk management

INDICATORS = [
    'atr',          # Average True Range (ATR) - for volatility tracking
    'boll_ub',      # Bollinger Upper Band - helps to identify overbought conditions
    'boll_lb',      # Bollinger Lower Band - helps to identify oversold conditions
    'rsi',          # Relative Strength Index (RSI) - measures momentum and overbought/oversold conditions
    'macd'          # Moving Average Convergence Divergence (MACD) - helps identify trends and reversals
]

In [62]:
print(INDICATORSS)

['atr', 'boll_ub', 'boll_lb', 'rsi', 'macd']


###Solving the previous error when directily called the FeatureEngineer function by just adding the INDICATORS 

In [ ]:
feature_engineer = FeatureEngineer(use_technical_indicator=True, tech_indicator_list=INDICATORSS) 
#Initializes FeatureEngineer to use specified technical indicators

In [ ]:
df = feature_engineer.add_technical_indicator(df) #Applies feature engineering to 'df' by adding the calculated technical indicators

In [ ]:
df = feature_engineer.add_turbulence(df) #adding the turbluance

In [78]:
df["date"] = pd.to_datetime(df["date"])  # Convert main dataset date column

In [79]:
stock_dimension = len(df.tic.unique())
state_space = 1 + 2*stock_dimension + len(INDICATORSS)*stock_dimension
print(f"Stock Dimension: {stock_dimension}, State Space: {state_space}")

Stock Dimension: 1631, State Space: 11418


In [80]:
env_kwargs = {
    "hmax": 100,
    "initial_amount": 1000000,
    "buy_cost_pct": 0.001,
    "sell_cost_pct": 0.001,
    "state_space": state_space,
    "stock_dim": stock_dimension,
    "tech_indicator_list": INDICATORSS,
    "action_space": stock_dimension,
    "reward_scaling": 1e-4,
    "print_verbosity":5

}

In [81]:
df.head()

,date,close,high,low,open,volume,tic,day,daily_return,atr,boll_ub,boll_lb,rsi,macd,turbulence
0,2004-01-01,0.105000,0.105000,0.105000,0.105000,0,DYA.TO,3,NaN,0.000000,NaN,NaN,NaN,0.0,0.0
1,2004-01-01,1.378000,1.378000,1.378000,1.378000,0,EDT.TO,3,12.123810,0.000000,NaN,NaN,NaN,0.0,0.0
2,2004-01-01,0.300000,0.300000,0.300000,0.300000,0,TSK.TO,3,-0.782293,0.000000,NaN,NaN,NaN,0.0,0.0
3,2004-01-02,8.244182,8.275001,8.205657,8.275001,18868,POU.TO,4,-0.842968,0.069344,NaN,NaN,NaN,0.0,0.0
4,2004-01-02,52.500000,52.500000,52.500000,52.500000,0,PNP.TO,4,39.152370,0.000000,NaN,NaN,NaN,0.0,0.0


In [82]:
df = df.sort_values(by='date').reset_index(drop=True)

In [83]:
split_idx = int(len(df) * 0.8)

In [84]:
# Extract training and validation periods directly
TRAIN_START_DATE = df.iloc[0]['date']
TRAIN_END_DATE = df.iloc[split_idx - 1]['date']

TEST_START_DATE = df.iloc[split_idx]['date']
TEST_END_DATE = df.iloc[-1]['date']
     

In [85]:
rebalance_window = 63 # rebalance_window is the number of days to retrain the model
validation_window = 63 # validation_window is the number of days to do validation and trading (e.g. if validation_window=63, then both validation and trading period will be 63 days)

ensemble_agent = DRLEnsembleAgent(df=df,
                 train_period=(TRAIN_START_DATE,TRAIN_END_DATE),
                 val_test_period=(TEST_START_DATE,TEST_END_DATE),
                 rebalance_window=rebalance_window,
                 validation_window=validation_window,
                 **env_kwargs)

In [86]:
SAC_model_kwargs = {
    "batch_size": 64,
    "buffer_size": 100000,
    "learning_rate": 0.0001,
    "learning_starts": 100,
    "ent_coef": "auto_0.1",
}
   

In [87]:
timesteps_dict = {'a2c' : 0,
                 'ppo' : 0,
                 'ddpg' : 0,
                 'sac' : 10_000,
                 'td3' : 0
                 }

In [88]:
df_summary = ensemble_agent.run_ensemble_strategy(
    A2C_model_kwargs={},  # Not using A2C
    PPO_model_kwargs={},  # Not using PPO
    DDPG_model_kwargs={},  # Using DDPG
    SAC_model_kwargs=SAC_model_kwargs,  # Not using SAC
    TD3_model_kwargs={},  # Not using TD3
    timesteps_dict=timesteps_dict
)

============Start Ensemble Strategy============
turbulence_threshold:  7680511933.831678
======Model training from:  2004-01-01 00:00:00 to  2022-10-18 00:00:00
======a2c Training========
{}
Using cpu device


ValueError: could not broadcast input array from shape (1650,) into shape (11418,)